# Executive B2B Energy Churn Prediction & Financial Risk Optimization

**Author:** Giovanni (Statistics Student, UCV)
**Role:** Quantitative Data Science Consultant

---

## Executive Overview
This notebook implements an end-to-end quantitative Machine Learning framework for predicting B2B customer churn at PowerCo. The methodology addresses severe class imbalance (~90% retention, 10% churn) and incorporates asymmetric financial loss functions to evaluate commercial discount strategies.

---

## Phase 1: Empirical Feature Engineering

### 1.1 Environment Setup & Data Loading
We import core computational libraries (`pandas`, `numpy`, `sklearn`) and load the raw datasets: clean client attributes, price time-series, and baseline static data.

In [1]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

df = pd.read_csv("clean_data_after_eda.csv")
price = pd.read_csv('price_data (1).csv')
client = pd.read_csv('client_data (1).csv')
dates = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
df[dates] = df[dates].apply(pd.to_datetime)
client[dates] = client[dates].apply(pd.to_datetime)

### 1.2 Initial Dataset Inspection
We inspect the top rows of `clean_data_after_eda`, `price_data`, and `client_data`.

In [2]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
print('Clear Data')
df.head()
print('Price Data')
price.head()
print('Client Data')
client.head()

Clear Data
Price Data
Client Data


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,0.0,1.78,0.114481,0.098142,40.606701,t,0.00,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,0.0,16.27,0.145711,0.000000,44.311378,f,0.00,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,0,0.0,38.72,0.165794,0.087899,44.311378,f,0.00,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,0,0.0,19.83,0.146694,0.000000,44.311378,f,0.00,30.22,30.22,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,526,0.0,131.73,0.116900,0.100015,40.606701,f,52.32,44.91,44.91,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0


### 1.3 Tariff Price Differences Calculation (December vs. January)
We filter price records for January (`2015-01-01`) and December (`2015-12-01`) and merge them on client `id`. We then perform vectorized matrix subtraction to obtain unit price deltas across off-peak, peak, and mid-peak periods for energy and power tariffs.

In [3]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
jan = price[price['price_date'] == '2015-01-01'].reset_index(drop=True)
dec = price[price['price_date'] == '2015-12-01'].reset_index(drop=True)
jan_dec = pd.merge(jan, dec, on='id', suffixes=('_jan', '_dec'))
variations = ['price_off_peak_var', 'price_peak_var', 'price_mid_peak_var', 'price_off_peak_fix', 'price_peak_fix', 'price_mid_peak_fix']
diff_cols = ['∆_price_off_peak_var', '∆_price_peak_var', '∆_price_mid_peak_var', '∆_price_off_peak_fix', '∆_price_peak_fix', '∆_price_mid_peak_fix']
jancols = [f'{v}_jan' for v in variations]
deccols = [f'{v}_dec' for v in variations]
jan_dec[diff_cols] = jan_dec[deccols].values - jan_dec[jancols].values
diff_cols = ['id'] + diff_cols
diff = jan_dec[diff_cols].copy()
diff.head()

,id,∆_price_off_peak_var,∆_price_peak_var,∆_price_mid_peak_var,∆_price_off_peak_fix,∆_price_peak_fix,∆_price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,-0.005508,0.000000,0.000000,0.177779,0.000000,0.000000
1,31f2ce549924679a3cbb2d128ae9ea43,-0.007221,-0.002324,0.003560,0.162916,0.097749,0.065166
2,36b6352b4656216bfdb96f01e9a94b4e,-0.010018,-0.005120,0.000763,0.162916,0.097749,0.065166
3,48f3e6e86f7a8656b2c6b6ce2763055e,-0.000488,0.000000,0.000000,0.177779,0.000000,0.000000
4,cce88c7d721430d8bd31f71ae686c91e,-0.039332,-0.033207,-0.028926,-0.335085,-0.400251,-0.432834


### 1.4 Total Financial Shock Computation ($\Delta \mathcal{S}$)
We merge tariff deltas with volumetric consumption (`cons_12m`) and contracted power (`pow_max`). Energy deltas are weighted by `cons_12m` and power deltas are weighted by `pow_max`. Summing these monetary additions yields the **Total Financial Shock** per customer.

In [4]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
var_cols = ['∆_price_off_peak_var', '∆_price_peak_var', '∆_price_mid_peak_var']
fixes = ['∆_price_off_peak_fix', '∆_price_peak_fix', '∆_price_mid_peak_fix']
cost = ['id', 'cons_12m', 'pow_max']
consum = pd.merge(client[cost], diff, on='id')
consum['cons_12m'] = consum['cons_12m'].astype(float)
consum[['add_cost_off_peak_var', 'add_cost_peak_var', 'add_cost_mid_peak_var']] = consum[var_cols].values * consum[['cons_12m']].values 
consum[['add_cost_off_peak_fix', 'add_cost_peak_fix', 'add_cost_mid_peak_fix']] = consum[fixes].values * consum[['pow_max']].values
costs = consum[['id', 'add_cost_off_peak_var', 'add_cost_peak_var', 'add_cost_mid_peak_var', 'add_cost_off_peak_fix', 'add_cost_peak_fix', 'add_cost_mid_peak_fix']].copy()
costs['total_financial_shock'] = costs.drop(columns='id').sum(axis=1)
costs[['id', 'total_financial_shock']].head()

,id,total_financial_shock
0,24011ae4ebbe3035111d65fa7c15bc57,-1609.083906
1,d29c2c54acc38ff3c0614d0a653813dd,-15.100873
2,764c75f661154dac3a6c254cd082ea7d,0.210055
3,bba03439a292a1e166f80264c16191cb,-4.855768
4,149d57cf92fc41cf94415803a877cb4b,-15.704517


### 1.5 Continuous Contract Tenure Calculation (in Months)
We calculate continuous months of relationship relative to cutoff date `2016-01-01` for activation (`act_months`), renewal (`ren_months`), and product modification (`modif_months`).

In [5]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
df_clean = pd.merge(df, costs[['id', 'total_financial_shock']], on='id')
df_clean['act_months'] = (2016 - df_clean['date_activ'].dt.year) * 12 + (1 - df_clean['date_activ'].dt.month)
df_clean['ren_months'] = (2016 - df_clean['date_renewal'].dt.year) * 12 + (1 - df_clean['date_renewal'].dt.month)
df_clean['modif_months'] = (2016 - df_clean['date_modif_prod'].dt.year) * 12 + (1 - df_clean['date_modif_prod'].dt.month)
df_clean[['act_months', 'ren_months', 'modif_months']].head()

,act_months,ren_months,modif_months
0,31,7,2
1,77,5,77
2,69,9,69
3,70,10,70
4,72,10,72


### 1.6 Categorical Encoding & Binarization
- `has_gas` mapped to 1 (`'t'`) and 0 (`'f'`).
- `channel_sales` and `origin_up` encoded via One-Hot Encoding (`pd.get_dummies`).

In [6]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
df_clean['has_gas'] = (df_clean['has_gas'] == 't').astype(int)
df_clean = pd.get_dummies(data=df_clean, columns=['channel_sales', 'origin_up'])
print('Engineered Dataset Shape:', df_clean.shape)

Engineered Dataset Shape: (14584, 60)


---
## Phase 2: Model Architecture & Training (Random Forest)

### 2.1 Target and Feature Matrix Separation & Stratified Split
We isolate target `churn` as `y` and drop non-predictive columns (`id`, raw date columns) to build feature matrix `x`. We apply `train_test_split` with `stratify=y` to preserve 90/10 imbalance.

### 2.2 Random Forest Fitting with Balanced Class Weights
We train `RandomForestClassifier` with `class_weight='balanced'` to penalize minority class errors.

In [7]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

y = df_clean['churn']
x = df_clean.drop(columns=['id', 'churn', 'date_activ', 'date_end', 'date_modif_prod', 'date_renewal'])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
model.fit(x_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

---
## Phase 3: Model Evaluation 

We evaluate model predictions against test labels using Confusion Matrix, Classification Report, and ROC-AUC.

In [8]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(x_test)
y_pred_proba = model.predict_proba(x_test)[:, 1]

# 1. Print Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 2. Print Classification Report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 3. Print ROC-AUC Score
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))

Confusion Matrix:
[[2600   34]
 [ 247   36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      2634
           1       0.51      0.13      0.20       283

    accuracy                           0.90      2917
   macro avg       0.71      0.56      0.58      2917
weighted avg       0.87      0.90      0.88      2917


ROC-AUC Score: 0.6926707287952328


---
## Phase 4: Asymmetric Financial Risk & Threshold Optimization

### 4.1 Financial Payoff Evaluation at Default Threshold (0.50)
PowerCo offers a 20% discount on `net_margin` to predicted churners:
- TP (Saved): $+0.80 \times \text{net\_margin}$
- FP (Giveaway): $-0.20 \times \text{net\_margin}$
- FN (Loss): $-1.00 \times \text{net\_margin}$

In [9]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
financial_df = pd.DataFrame({
    'net_margin': x_test['net_margin'],
    'actual': y_test,
    'predicted': y_pred
})

tp = financial_df[(financial_df['actual'] == 1) & (financial_df['predicted'] == 1)]
fp = financial_df[(financial_df['actual'] == 0) & (financial_df['predicted'] == 1)]
fn = financial_df[(financial_df['actual'] == 1) & (financial_df['predicted'] == 0)]

saved = 0.80 * tp['net_margin'].sum()
giveaway = 0.20 * fp['net_margin'].sum()
losses = 1.00 * fn['net_margin'].sum()

print(f"Saved money (TP): ${saved:.2f}")
print(f"Accidental giveaway (FP): ${giveaway:.2f}")
print(f"Losses (FN): ${losses:.2f}")
print(f"Net Financial Impact: ${saved - giveaway - losses:.2f}")

Saved money (TP): $5543.34
Accidental giveaway (FP): $1482.73
Losses (FN): $54439.25
Net Financial Impact: $-50378.64


### 4.2 Financial Evaluation at 20% Threshold (0.20)

In [10]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
y_pred_20 = (y_pred_proba >= 0.20).astype(int)

financial_20 = pd.DataFrame({
    'net_margin': x_test['net_margin'],
    'actual': y_test,
    'predicted': y_pred_20
})

tp_20 = financial_20[(financial_20['actual'] == 1) & (financial_20['predicted'] == 1)]
fp_20 = financial_20[(financial_20['actual'] == 0) & (financial_20['predicted'] == 1)]
fn_20 = financial_20[(financial_20['actual'] == 1) & (financial_20['predicted'] == 0)]

saved_20 = 0.80 * tp_20['net_margin'].sum()
giveaway_20 = 0.20 * fp_20['net_margin'].sum()
losses_20 = 1.00 * fn_20['net_margin'].sum()
net_impact_20 = saved_20 - giveaway_20 - losses_20

print(f"--- Threshold 20% (0.20) ---")
print(f"Saved Money (TP): +${saved_20:.2f}")
print(f"Accidental Giveaway (FP): -${giveaway_20:.2f}")
print(f"Unseen Churn Losses (FN): -${losses_20:.2f}")
print(f"Net Financial Impact: ${net_impact_20:.2f}")

--- Threshold 20% (0.20) ---
Saved Money (TP): +$32933.95
Accidental Giveaway (FP): -$37044.72
Unseen Churn Losses (FN): -$20200.99
Net Financial Impact: $-24311.76


### 4.3 Optimal Decision Threshold Sweep ($p \in [0.10, 0.80]$)

In [11]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
results_sweep = []
for th in np.arange(0.10, 0.85, 0.05):
    y_pred_th = (y_pred_proba >= th).astype(int)
    fin = pd.DataFrame({'net_margin': x_test['net_margin'], 'actual': y_test, 'predicted': y_pred_th})
    
    tp = fin[(fin['actual'] == 1) & (fin['predicted'] == 1)]['net_margin'].sum()
    fp = fin[(fin['actual'] == 0) & (fin['predicted'] == 1)]['net_margin'].sum()
    fn = fin[(fin['actual'] == 1) & (fin['predicted'] == 0)]['net_margin'].sum()
    
    saved = 0.80 * tp
    giveaway = 0.20 * fp
    losses = 1.00 * fn
    net = saved - giveaway - losses
    
    results_sweep.append({
        'Threshold': round(th, 2),
        'Saved_TP': round(saved, 2),
        'Giveaway_FP': round(giveaway, 2),
        'Losses_FN': round(losses, 2),
        'Net_Impact': round(net, 2)
    })

sweep_df = pd.DataFrame(results_sweep)
sweep_df.sort_values(by='Net_Impact', ascending=False)

,Threshold,Saved_TP,Giveaway_FP,Losses_FN,Net_Impact
3,0.25,28813.78,21345.54,25351.21,-17882.97
4,0.30,23652.99,14008.52,31802.19,-22157.72
2,0.20,32263.86,34284.01,21038.61,-23058.77
1,0.15,37734.82,49010.22,14199.90,-25475.29
0,0.10,44123.67,69896.13,6213.84,-31986.30
5,0.35,11839.25,6826.69,46569.37,-41556.81
6,0.40,8314.63,4185.47,50975.14,-46845.98
7,0.45,6960.95,2657.96,52667.24,-48364.25
8,0.50,5543.34,1482.73,54439.25,-50378.64
9,0.55,4140.34,1183.63,56193.01,-53236.30


### 4.4 Feature Importances & Econometric Elasticity Audit

In [12]:
# pyright: reportUndefinedVariable=false, reportMissingTypeStubs=false
importances = pd.Series(model.feature_importances_, index=x.columns)
top_10 = importances.sort_values(ascending=False).head(10)
print("Top 10 Feature Importances:")
print(top_10)

Top 10 Feature Importances:
margin_gross_pow_ele       0.059661
margin_net_pow_ele         0.058515
cons_12m                   0.056892
total_financial_shock      0.055100
forecast_meter_rent_12m    0.054862
net_margin                 0.048748
forecast_cons_12m          0.048091
cons_last_month            0.039170
var_year_price_off_peak    0.037401
act_months                 0.037098
dtype: float64
